In [46]:
import dill
import os
import numpy as np
import jax
import jax.numpy as jnp
from jax import jit
from functools import partial

# Load true funcitons
ex_name = "ex8"
file_path = os.path.join("true_functions", f"{ex_name}.pkl")

with open(file_path, "rb") as f:
    loaded_functions = dill.load(f)

true_drift = loaded_functions["drift"]
true_diffusion = loaded_functions["diffusion"]
true_drift = jax.jit(true_drift)
true_diffusion = jax.jit(true_diffusion)

n_dimensions = loaded_functions["n_dimensions"]
diff_type = loaded_functions["diff_type"]
xlim = loaded_functions["xlim"]


In [47]:
# training data parameters
n_trajectories = 100000
trajectory_time = 1e-4
step_size = 1e-6
grid_resolution = round(trajectory_time/step_size) + 1 # +1 to account for staring point

step_sizes = np.zeros((n_trajectories, grid_resolution-1)) + step_size

key = jax.random.PRNGKey(0)

In [48]:
@partial(jit, static_argnames=("drift_fn", "diffusion_fn", "grid_resolution"))
def simulate_euler_maruyama(key, drift_fn, diffusion_fn, x0, h, grid_resolution):
    """
    Simulate trajectories using Euler-Maruyama.
    - x0: (N, D) array of initial positions
    - Returns: x: (N, D, T)
    """
    n_trajectories, D = x0.shape
    T = grid_resolution

    # Generate dW: shape (T-1, N, D)
    key, subkey = jax.random.split(key)
    dW = jax.random.normal(subkey, shape=(T-1, n_trajectories, D)) * jnp.sqrt(h)

    def euler_step(x_t, dW_t):
        drift = drift_fn(x_t)                 # (N, D)
        diffusion = diffusion_fn(x_t).reshape(x_t.shape[0], D, D)         # (N, D, D)
        diffusion_term = jnp.einsum('nij,nj->ni', diffusion, dW_t)  # (N, D)
        #diffusion_term = jnp.matmul(diffusion, dW_t[..., None]).squeeze(-1)
        x_next = x_t + drift * h + diffusion_term
        return x_next, x_next

    # Run lax.scan
    _, xs = jax.lax.scan(euler_step, x0, dW)
    
    # Add initial condition to beginning
    x_full = jnp.concatenate([x0[:, :, None], xs.transpose(1, 2, 0)], axis=-1)
    r_data = jnp.diff(x_full, axis=-1) # (N, D, T)
    x_data = x_full[:, :, :-1]
    
    return x_data, r_data

In [49]:
# trajectory start points
x0 = jax.random.uniform(key, shape=(n_trajectories, xlim.shape[0]), minval=xlim[:,0], maxval=xlim[:,1])

# generate trajectories
x_data, r_data = simulate_euler_maruyama(key,
                            drift_fn=true_drift,
                            diffusion_fn=true_diffusion,
                            x0=x0,
                            h=step_size,
                            grid_resolution=grid_resolution)


In [50]:
# def simulate_euler_maruyama_coupled(key, drift_fn, diffusion_fn, coupled_fn, y0, x0, h, grid_resolution):
#     """
#     Simulate trajectories using Euler-Maruyama.
#     - y0: (N, D) initial state for the process you want to evolve
#     - x0: (N, Dx) input variables for drift/diffusion (can be same as y0 or different)
#     - Returns: y: (N, D, T)
#     """
#     n_trajectories, D = y0.shape
#     T = grid_resolution

#     # Generate dW: shape (T-1, N, D)
#     key, subkey = jax.random.split(key)
#     dW = jax.random.normal(subkey, shape=(T-1, n_trajectories, D)) * jnp.sqrt(step_size)

#     def euler_step(carry, dW_t):
#         y_t, x_t = carry
        
#         x_next = coupled_fn(y_t, x_t, h)

#         drift = drift_fn(x_next)                                # (N, D)
#         diffusion = diffusion_fn(x_next).reshape(y_t.shape[0], D, D)  # (N, D, D)
#         diffusion_term = jnp.einsum("nij,nj->ni", diffusion, dW_t)     # (N, D)

#         y_next = y_t + drift * h + diffusion_term
#         return (y_next, x_next), (y_next, x_next)

#     carry0 = (y0, x0)
#     _, trajs = jax.lax.scan(euler_step, carry0, dW)
#     y_hist, x_hist = trajs
    
#     # Add initial condition to beginning
#     y_full = jnp.concatenate([y0[:, :, None], y_hist.transpose(1, 2, 0)], axis=-1) # (N,D,T)
#     r_data = jnp.diff(y_full, axis=-1)
#     x_data = x_hist.transpose(1, 2, 0)
    
#     return x_data, r_data


In [51]:
# # coupled
# coupled_fn = loaded_functions["coupled_fn"]

# # trajectory start points
# x0 = jax.random.uniform(key, shape=(n_trajectories, xlim.shape[0]), minval=xlim[:,0], maxval=xlim[:,1])
# y0 = x0[:,0:1]

# # generate trajectories
# x_data, r_data = simulate_euler_maruyama_coupled(key,
#                             drift_fn=true_drift,
#                             diffusion_fn=true_diffusion,
#                             coupled_fn=coupled_fn, 
#                             y0=y0,
#                             x0=x0,
#                             h=step_size,
#                             grid_resolution=grid_resolution)


In [52]:
print(x_data.shape, r_data.shape, step_sizes.shape)

(100000, 2, 100) (100000, 2, 100) (100000, 100)


In [53]:

np.savez(f"training_data/{ex_name}.npz", x_data=np.array(x_data), r_data=np.array(r_data), step_sizes=step_sizes, diff_type=diff_type)
